In [60]:
import requests
from datetime import datetime
import pandas as pd
import time

BASE = "https://api.openf1.org/v1"

In [61]:
def get_races(year: int):
    sessions = requests.get(f"{BASE}/sessions", params = {"year": year}).json()

    return sessions

In [ ]:
# Sessions Schema
years = [2023, 2024, 2025] # 2023 is the earliest available in the API
sessions = []

for year in years:
    sessions.extend(get_races(year))

# Construct Sessions Table
session_df = pd.DataFrame(sessions)

# drop unncessary columns: meeting_key, location, country_key, country_code, country_name, circuit_short_name, gmt_offset
session_df = session_df.drop(columns=["meeting_key", "location", "country_key", "country_code", "country_name", "circuit_short_name", "gmt_offset"])

# reorder columns to match schema: session_key, session_type, session_name, circuit_key, date_start, date_end, year
session_df = session_df[["session_key", "session_type", "session_name", "circuit_key", "date_start", "date_end", "year"]]

# check for missing values
print(session_df.isnull().sum())

# save to csv
session_df.to_csv("../../data/cleaned/sessions.csv", index=False)

session_key     0
session_type    0
session_name    0
circuit_key     0
date_start      0
date_end        0
year            0
dtype: int64


In [76]:
def get_starting_grid(session_key: int, position: int):
    # sample curl "https://api.openf1.org/v1/starting_grid?session_key=7783&position%3C=3"
    # sample url: https://api.openf1.org/v1/starting_grid?session_key=7783&position<=3
    grid = requests.get(f"{BASE}/starting_grid", params = {"session_key": session_key, "position<": position}).json()

    return grid

In [78]:
# Starting Grid Schema
starting_grids = []
position = 20

for session_key in session_df["session_key"]:
    # starting grid only exists for qualifying sessions
    if session_df.loc[session_df["session_key"] == session_key, "session_type"].values[0] == "Qualifying":
        starting_grids.extend(get_starting_grid(session_key, position = position))

        # add sleep to avoid rate limiting
        time.sleep(1)
  
# Construct Starting Grid Table
grid_df = pd.DataFrame(starting_grids)

# drop unnecessary columns: meeting_key
grid_df = grid_df.drop(columns=["meeting_key"])

# reorder columns to match schema: session_key, driver_number, position, lap_duration
grid_df = grid_df[["session_key", "driver_number", "position", "lap_duration"]]

# check for missing values
print(grid_df.isnull().sum())

# lap_duration can be null for drivers who DNF, DNS, or DSQ prior to finishing the first lap
# for now, we will keep it as null as we want to be able to distinguish between the slowest drivers
# and those who did not finish the session

# save to csv
grid_df.to_csv("../../data/cleaned/starting_grid.csv", index=False)



session_key       0
driver_number     0
position          0
lap_duration     64
dtype: int64


In [64]:
def get_race_result(session_key: int, position: int):
    # sample curl "https://api.openf1.org/v1/session_result?session_key=7782&position%3C=3"
    # sample url: https://api.openf1.org/v1/session_result?session_key=7782&position<=3
    result = requests.get(f"{BASE}/session_result", params = {"session_key": session_key, "position<": position}).json()

    return result

In [72]:
# Race Result Schema
race_results = []
position = 20

for session_key in session_df["session_key"]:
    if session_df.loc[session_df["session_key"] == session_key, "session_type"].values[0] == "Race":
        race_results.extend(get_race_result(session_key, position = position))

        # add sleep to avoid rate limiting
        time.sleep(1)

# Construct Race Result Table
race_result_df = pd.DataFrame(race_results)

# drop unnecessary columns: meeting_key, points, duration, gap_to_leader, 
race_result_df.drop(columns=["meeting_key", "points", "duration", "gap_to_leader", "number_of_laps"], inplace=True)

# go through dnf, dns and dsq, if one is true, set driver_status to the first one that is true, if none are true, set to finished
def determine_driver_status(row):
    if row.get('dnf', False):
        return "dnf"
    elif row.get('dns', False):
        return "dns"
    elif row.get('dsq', False):
        return "dsq"
    else:
        return "finished"

race_result_df['driver_status'] = race_result_df.apply(determine_driver_status, axis=1)

# we no longer need the dnf, dns and dsq columns
race_result_df.drop(columns=["dnf", "dns", "dsq"], inplace=True)

# reorder columns to match schema: session_key, driver_number, position, driver_status
race_result_df = race_result_df[["session_key", "driver_number", "position", "driver_status"]]

# check for missing values
print(race_result_df.isnull().sum())

session_key      0
driver_number    0
position         0
driver_status    0
dtype: int64


In [73]:
# save to csv
race_result_df.to_csv("../../data/cleaned/race_result.csv", index=False)

In [85]:
# Circuit Schema
circuits_df = pd.DataFrame(sessions) # sessions is from the get_sessions function

# keep only circuit_key, country_name, country_code
circuits_df = circuits_df[["circuit_key", "country_name", "country_code"]]

# check for missing values
print(circuits_df.isnull().sum())

# save to csv
circuits_df.to_csv("../../data/cleaned/circuits.csv", index=False)

circuit_key     0
country_name    0
country_code    0
dtype: int64


In [119]:
def get_driver(session_key: int, driver_number: int):
    # sample curl "https://api.openf1.org/v1/drivers?driver_number=1&session_key=9158"
    # sample url: https://api.openf1.org/v1/drivers?driver_number=1&session_key=9158
    driver = requests.get(f"{BASE}/drivers", params = {"driver_number": driver_number, "session_key": session_key}).json()

    return driver

def get_drivers(session_key: int):
    drivers = requests.get(f"{BASE}/drivers", params = {"session_key": session_key}).json()

    return drivers

In [ ]:
# Drivers Schema
drivers = []
num_sessions = 20

# for session_key in session_df["session_key"][:num_sessions]:
#     # find all drivers in the session using race_result_df
#     drivers_in_session = race_result_df[race_result_df["session_key"] == session_key]["driver_number"].unique()

#     for driver_number in drivers_in_session:
#         drivers.append(get_driver(session_key, driver_number))

#         # add sleep to avoid rate limiting
#         time.sleep(1)

for session_key in session_df["session_key"]:
    if session_df.loc[session_df["session_key"] == session_key, "session_type"].values[0] == "Race":
        drivers.extend(get_drivers(session_key))

        # add sleep to avoid rate limiting
        time.sleep(1)
    
# Construct Drivers Table
drivers_df = pd.DataFrame(drivers)

# keep only session_key, driver_number, first_name, last_name, team_name
drivers_df = drivers_df[["session_key", "driver_number", "first_name", "last_name", "team_name"]]

# check for missing values
print(drivers_df.isnull().sum())

# save to csv
drivers_df.to_csv("../../data/cleaned/drivers.csv", index=False)

session_key       0
driver_number     0
first_name       15
last_name        15
team_name        15
dtype: int64
